# Follow an output back to its original inputs

Pick one result element and follow the index, transpose and sum that produced it. A repeated source contributes more than once, even when it is drawn as one cell.

This notebook also works without notebook controls. Static figures and coordinate traces remain available.

In [ ]:
import numpy as np
from IPython.display import display

import rainbow_tensor as rt

x = np.arange(1, 7).reshape(2, 3)
flow = rt.Flow()
source = flow.input(x, name="X")
display(x)

## Predict the selected rows

The index chooses rows **1, 0, 1** in that order. The reverse slice reads each chosen row from right to left.

Before running the next cell, predict the first and last rows of `S`. Do they come from different source rows?

In [ ]:
selection = ([1, 0, 1], slice(None, None, -1))
selected = flow.index(source, selection, name="S")
transposed = flow.transpose(selected, name="T")
y = flow.sum(transposed, axis=1, name="Y")

assert selected.shape == (3, 3)
assert transposed.shape == (3, 3)
assert y.shape == (3,)
assert [y.value((i,)) for i in range(3)] == [15, 12, 9]
display(y.visualize(focus=(0,)))

## Three contributions from two original positions

`Y[0]` sums the first row of `T`: `6 + 3 + 6 = 15`.

Following the transpose and index gives `X[1, 2] + X[0, 2] + X[1, 2]`. The second source row was sampled twice. That produces two separate contribution paths to `X[1, 2]`.

The immutable trace below records coordinates without reading any values.

In [ ]:
trace = y.trace((0,))
assert trace.complete
counts = {root.reference.coordinate: root.count for root in trace.roots}
assert counts == {(1, 2): 2, (0, 2): 1}

for coordinate, count in sorted(counts.items()):
    print(f"X{coordinate} contributes {count} time(s)")

## Choose a different result

Predict the original coordinates and value for `Y[1]` before selecting it. Then try `Y[2]`.

Click a final `Y` cell. You can also focus a cell with the keyboard and press Enter or Space. Coordinate fields reach positions outside a large preview.

Install `rainbow-tensor` in this notebook's kernel environment. Live controls require a running kernel and widget support in its host. For hosts without widget support, use `display(y.visualize(focus=(1,)))`.

In [ ]:
explorer = rt.explore(y)
display(explorer)

In [ ]:
explorer.set_focus((1,))
focused = explorer.visual
assert focused.result_shape == (3,)
assert focused.trace.output_coord == (1,)
assert y.value((1,)) == 12
display(focused)

## Follow a single shape operation

The same focus convention also works without a Flow. A reshape preserves row-major position while changing its coordinate.

Predict the source coordinate of result `(2, 1)` when a `(2, 3)` input becomes `(3, 2)`. It is the last element in both layouts.

In [ ]:
reshaped = rt.reshape(x, (3, 2), focus=(2, 1))
assert reshaped.trace.terms[0][0].coordinate == (1, 2)
display(reshaped)

# For a clickable version, run rt.explore(rt.reshape, x, (3, 2)).

## Keep each mean in its own step

A nested mean keeps its intermediate grouping. Each row mean divides by three, then the final mean divides by two. Root participation counts describe paths, not numerical weights.

In [ ]:
row_means = flow.mean(source, axis=1, name="RowMeans")
overall_mean = flow.mean(row_means, name="OverallMean")
assert overall_mean.value(()) == 3.5
mean_trace = overall_mean.trace(())
assert mean_trace.complete
assert mean_trace.steps[0].divisor == 2
assert row_means.divisor == 3
display(overall_mean.visualize(focus=()))

## Values refresh while recipes stay fixed

The input array remains live. Changing one value affects both paths that read it on the next query. Recorded axes, shapes and selection parameters stay fixed. Build a new step to use a different index, and keep input shapes unchanged.

In [ ]:
x[1, 2] = 60
assert y.value((0,)) == 123  # 60 + 3 + 60
x[1, 2] = 6
assert y.value((0,)) == 15

## Know when an explanation is incomplete

Depth, occurrence and edge limits keep traces small. An incomplete trace reports what was omitted. Its original-input counts include only paths it reached.

Numerical limits are separate. Planning counts recursive terms, factor references and input reads before it reads any array values. A value request raises `ValueBudgetExceeded` when it cannot fit. A visual shows question marks throughout its panels instead of a partial calculation.

In [ ]:
partial = y.trace((0,), max_depth=1)
assert not partial.complete
assert "max_depth" in partial.truncated_reasons
print("Trace stopped at:", partial.truncated_reasons)

try:
    y.value((0,), max_terms=2)
except rt.ValueBudgetExceeded as error:
    assert error.reason == "max_terms"
    print("Value planning stopped at:", error.reason)

bounded = y.visualize(focus=(0,), max_terms=2)
assert bounded.metadata["value_evaluation"]["status"] == "skipped"
display(bounded)

The normal defaults allow 10,000 terms per required element and 100,000 total units of recursive work. Traces allow depth 6, 80 occurrences and 120 edges. Figures show at most 8 reached tensors. Numerical recursion has a separate depth limit of 64.

Values use Python scalar arithmetic and can differ from a framework's native dtype, rounding or overflow rules. No intermediate tensor or backend kernel is created by the Flow.

When finished with the live explorer, run `explorer.close()` if it is not `None`. The latest static result remains in `focused` and can be saved with `focused.save("operation-origins.svg")`.

Read [the provenance guide](../docs/guide/provenance.md) for broadcast outputs, parameter snapshots and exact limit definitions.